# TimeDataModel — Walkthrough

This notebook walks through the core features of `timedatamodel`: a domain-agnostic
time series data model with rich metadata, geolocation support, and seamless
conversion to NumPy, pandas, and polars.

## 1. Enums

`timedatamodel` provides three `StrEnum` types that serialize naturally to strings.

In [ ]:
from timedatamodel import Frequency, DataType, StorageType

# Frequency uses ISO 8601 duration strings
print("Hourly:", Frequency.PT1H)
print("Daily: ", Frequency.P1D)
print("Monthly:", Frequency.P1M)

# StrEnum — direct string comparison works
assert Frequency.PT1H == "PT1H"

# DataType classifies the nature of the data
print("\nAll data types:", [dt.value for dt in DataType])

# StorageType describes how overlapping instances are stored
print("Storage types:", [st.value for st in StorageType])

## 2. Location

Two location types cover point and area use cases:
- **`GeoLocation`** — a single (latitude, longitude) point
- **`GeoArea`** — a Shapely polygon with optional name

In [ ]:
from timedatamodel import GeoLocation, GeoArea

# A point location — Oslo, Norway
oslo = GeoLocation(latitude=59.91, longitude=10.75)
print("Oslo:", oslo)

In [ ]:
# Coordinates are validated on construction
try:
    GeoLocation(latitude=999, longitude=0)
except ValueError as e:
    print("Caught:", e)

In [ ]:
# A polygon area — simplified southern Norway region
# Coordinates are (latitude, longitude) pairs
no1 = GeoArea.from_coordinates(
    [(58.0, 6.0), (58.0, 12.0), (62.0, 12.0), (62.0, 6.0)],
    name="NO1",
)

print("Area name:  ", no1.name)
print("Bounding box:", no1.bounds)  # (min_lon, min_lat, max_lon, max_lat)
print("Centroid:   ", no1.centroid)

## 3. Resolution

`Resolution` pairs a `Frequency` with a timezone. Timezones are validated
against the standard library's `zoneinfo`.

In [ ]:
from timedatamodel import Resolution

hourly_cet = Resolution(frequency=Frequency.PT1H, timezone="Europe/Oslo")
daily_utc = Resolution(frequency=Frequency.P1D)  # defaults to UTC
monthly = Resolution(frequency=Frequency.P1M)

print("Hourly CET:", hourly_cet)
print("ZoneInfo:  ", hourly_cet.tz)
print("Timedelta: ", hourly_cet.to_timedelta())
print()
print("Monthly is calendar-based:", monthly.is_calendar_based)
print("Monthly timedelta:        ", monthly.to_timedelta())  # None for calendar-based

## 4. Metadata

`Metadata` bundles descriptive information about a time series. Units are stored as
plain strings (easy to serialize) and resolved to `pint.Unit` on demand via the
`pint_unit` property.

In [ ]:
from timedatamodel import Metadata

meta = Metadata(
    unit="MW",
    data_type=DataType.ACTUAL,
    location=oslo,
    name="power",
    description="Hourly power generation",
    attributes={"source": "example", "fuel": "wind"},
)

print("Unit string:", meta.unit)
print("Pint unit:  ", meta.pint_unit)
print("Data type:  ", meta.data_type)
print("Location:   ", meta.location)
print("Attributes: ", meta.attributes)

## 5. Creating a TimeSeries

A `TimeSeries` combines a `Resolution`, `Metadata`, and the actual data.
It can be constructed from parallel lists or from a list of `DataPoint` tuples.

In [ ]:
from datetime import datetime, timedelta, timezone
from timedatamodel import TimeSeries, DataPoint

# Construction from parallel lists
base = datetime(2024, 6, 1, tzinfo=timezone.utc)
timestamps = [base + timedelta(hours=i) for i in range(24)]
values = [120.0 + i * 3.5 for i in range(24)]
values[5] = None  # simulate a missing reading

ts = TimeSeries(
    hourly_cet,
    meta,
    timestamps=timestamps,
    values=values,
)

# In Jupyter, just placing the object at the end of a cell renders a rich display
ts

In [ ]:
# Construction from DataPoint named tuples
data_points = [
    DataPoint(datetime(2024, 1, 1, tzinfo=timezone.utc), 10.0),
    DataPoint(datetime(2024, 1, 1, 1, tzinfo=timezone.utc), 12.5),
    DataPoint(datetime(2024, 1, 1, 2, tzinfo=timezone.utc), None),
]

ts2 = TimeSeries(hourly_cet, data=data_points)
ts2

## 6. Sequence Protocol

`TimeSeries` supports `len()`, indexing, slicing, iteration, and truthiness.

In [ ]:
# Indexing returns a DataPoint named tuple
first = ts[0]
print(f"First point: {first.timestamp} -> {first.value}")

# Slicing returns a list of DataPoints
print(f"\nFirst 3 points:")
for dp in ts[:3]:
    print(f"  {dp.timestamp}  {dp.value}")

# Iteration
print(f"\nLast 3 points:")
for dp in ts[-3:]:
    print(f"  {dp.timestamp}  {dp.value}")

# Truthiness
print(f"\nHas data: {bool(ts)}")
print(f"Empty:    {bool(TimeSeries(hourly_cet))}")

## 7. Export to NumPy

`to_numpy()` returns values as a `float64` array where `None` becomes `NaN`.

In [ ]:
import numpy as np

arr = ts.to_numpy()
print(f"Shape: {arr.shape}, dtype: {arr.dtype}")
print(f"Values: {arr[:8]}")
print(f"Contains NaN at index 5: {np.isnan(arr[5])}")

## 8. Export to pandas

`to_pandas_dataframe()` returns a `DataFrame` with a `DatetimeIndex`. The column
name comes from `metadata.name` (or defaults to `"value"`).

In [ ]:
df_pd = ts.to_pandas_dataframe()
print(f"Columns: {df_pd.columns.tolist()}")
print(f"Index type: {type(df_pd.index).__name__}")
print()
df_pd.head(8)

## 9. Export to polars

`to_polars_dataframe()` returns a polars `DataFrame` with `"timestamp"` and value
columns. polars is lazy-imported, so it's only needed if you call this method.

In [ ]:
df_pl = ts.to_polars_dataframe()
print(f"Columns: {df_pl.columns}")
print()
df_pl.head(8)

## 10. Round-Trip Conversion

You can convert back from pandas or polars DataFrames to a `TimeSeries`.

In [ ]:
# pandas round-trip
ts_from_pd = TimeSeries.from_pandas(df_pd, hourly_cet, meta)
print(f"From pandas: {len(ts_from_pd)} points")
print(f"  First: {ts_from_pd[0]}")
print(f"  Missing value preserved: {ts_from_pd[5].value is None}")

# polars round-trip
ts_from_pl = TimeSeries.from_polars(df_pl, hourly_cet, meta)
print(f"\nFrom polars: {len(ts_from_pl)} points")
print(f"  First: {ts_from_pl[0]}")
print(f"  Missing value preserved: {ts_from_pl[5].value is None}")

## 11. Validation

`validate()` checks for common issues: timestamp ordering and frequency consistency.

In [ ]:
# Our time series is well-formed
print("Valid series:", ts.validate())

# Create one with problems
bad_ts = TimeSeries(
    hourly_cet,
    timestamps=[
        base + timedelta(hours=2),  # out of order
        base + timedelta(hours=1),
        base + timedelta(hours=5),  # gap (inconsistent frequency)
    ],
    values=[1.0, 2.0, 3.0],
)

print("\nBad series warnings:")
for w in bad_ts.validate():
    print(f"  - {w}")

## 12. GeoArea Example

For area-based data (e.g., price zones, weather regions), use `GeoArea` as the
location in metadata.

In [ ]:
area_meta = Metadata(
    unit="EUR/MWh",
    data_type=DataType.ACTUAL,
    location=no1,
    name="price",
    description="Day-ahead electricity price for NO1",
)

daily_prices = TimeSeries(
    Resolution(frequency=Frequency.P1D),
    area_meta,
    timestamps=[base + timedelta(days=i) for i in range(7)],
    values=[45.2, 47.8, 42.1, 50.3, 48.9, 44.6, 46.7],
)

daily_prices